## Imports

In [1]:
import os
import random
import torch
import numpy as np

from fairface_vit       import FairFaceViT
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import get_train_transform, get_age_transform, get_val_transform

from training.trainer   import evaluate_loop
from training.losses    import get_age_weights, get_race_weights, get_loss_function
from training.checkpoint_utils import build_model_from_checkpoint

/home/ashkanrn/01-Project/University/Deep-Learning/DL-project/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Seed

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Model selection

In [3]:
# change ONLY this + run_version to evaluate a different trained model
model_name = "clip"  # "clip" | "dinov2" | "siglip"
run_version = "v3"
 
if model_name == "clip":
    from model.clip import model as backbone_model, processor as backbone_processor
 
elif model_name == "dinov2":
    from model.dinov2 import model as backbone_model, processor as backbone_processor
 
elif model_name == "siglip":
    from model.siglip import model as backbone_model, processor as backbone_processor
 
else:
    raise ValueError(f"Unknown model_name: {model_name}")
 
RUN_NAMES = {
    "clip"  : "clip-vit-base-patch16",
    "dinov2": "dinov2-vit-base",
    "siglip": "siglip-vit-base-patch16",
}
 
run_name = RUN_NAMES[model_name]
 
checkpoint_dir  = f"../checkpoints/{model_name}/{run_version}"
best_heads_path = f"{checkpoint_dir}/{model_name}-best-heads.pt"
 
if not os.path.exists(best_heads_path):
    raise FileNotFoundError(f"No best-heads checkpoint found at {best_heads_path}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13888.42it/s]
[transformers] CLIPVisionModel LOAD REPORT from: ../models/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_pr

## Hyperparameters

In [ ]:
# # must match the values used to build the head architecture during training
# age_dropout1   = 0.15
# age_hidden_dim = 384
# batch_size     = 20
 
# # only used to instantiate the loss functions evaluate_loop expects;
# # doesn't affect reported accuracy/f1/mae metrics


## Model and device

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load checkpoint and rebuild model (self-configures age_loss_type + adapter method)
checkpoint = torch.load(best_heads_path, map_location=device, weights_only=False)
fairface_model, age_loss_type, adapter_cfg = build_model_from_checkpoint(
    checkpoint, backbone_model, device=device
)

 
print(
    f"Loaded {model_name}-best-heads.pt | "
    f"trained epoch={checkpoint['epoch']} | "
    f"best val weighted F1={checkpoint['best_acc']:.4f} | "
    f"age_loss_type={age_loss_type} | "
    f"adapter={adapter_cfg.method}"
)

Loaded clip-best-heads.pt | trained epoch=8 | best val weighted F1=0.7414 | age_loss_type=corn | adapter=lora


## Dataset and dataloaders

In [5]:
train_set, val_set, test_set = get_dataset(
    get_train_transform(backbone_processor),
    get_age_transform(backbone_processor),
    get_val_transform(backbone_processor),
)

_, val_dataloder, test_loader = get_dataloaders(
    train_set,
    val_set,
    test_set
)

load_dataset: 30.49s
extract splits: 0.00s
prepare indices: 0.00s
train_test_split: 0.27s
select(): 0.03s


## Loss functions

In [6]:
race_labels = np.array(train_set.dataset["race"])
race_weights = get_race_weights(race_labels, num_classes=7, device=device)

if age_loss_type == "ce":
    age_labels = np.array(train_set.dataset["age"])
    age_weights = get_age_weights(age_labels, num_classes=9, device=device)
else:
    age_weights = None

loss_funcs = get_loss_function(
    age_loss_type=age_loss_type,
    age_weights=age_weights,
    race_weights=race_weights,
    num_age_classes=9,
)

## Final evaluation on held-out test set

In [7]:
LOSS_WEIGHT = {
    "gender": 1,
    "age":    1,
    "race":   1
}

test_loss, test_task_loss, test_metrics, test_subgroup_metrics = evaluate_loop(
    val_dataloder, fairface_model, loss_funcs, LOSS_WEIGHT, device,
    epoch=1, epochs=1, use_amp=False
)
 
avg_accuracy = (
    test_metrics["gender"]["accuracy"]
    +
    test_metrics["age"]["accuracy"]
    +
    test_metrics["race"]["accuracy"]
) / 3

task_weighted_f1 = (
    (0.25 * test_metrics['gender']['f1'])
    + (0.4 * test_metrics['age']['f1'])
    + (0.35 * test_metrics['race']['f1'])
)
 
print("\n=== TEST SET RESULTS ===")
print(f"model: {run_name} | run_version: {run_version}\n")
print(f"avg loss: {test_loss:.4f}")
print(f"avg accuracy (3-task mean): {avg_accuracy:.4f}")
print(f"task weighted f1: {task_weighted_f1:.4f}\n")

for task, metrics in test_metrics.items():
    print(f"\n{task}: ")

    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


print("\n--- subgroup metrics ---")
for group_name, values in test_subgroup_metrics.items():
    print(f"{group_name}:")
    for subgroup, acc in values.items():
        print(f"  {subgroup}: {acc:.3f}")

validation Epoch 1/1:   0%|          | 0/274 [00:00<?, ?it/s]

-VALIDATION
 avg loss: 1.0563
 gender: acc=0.957  f1=0.954
 age:    acc=0.626  f1=0.600   mae=0.406
 race:   acc=0.753  f1=0.751

=== TEST SET RESULTS ===
model: clip-vit-base-patch16 | run_version: v3

avg loss: 1.0563
avg accuracy (3-task mean): 0.7787
task weighted f1: 0.7413


gender: 
accuracy: 0.9571
precision: 0.9647
recall: 0.9434
f1: 0.9539

age: 
accuracy: 0.6259
precision: 0.6465
recall: 0.5741
f1: 0.6001
mse: 0.4868
rmse: 0.6977
mae: 0.4061

race: 
accuracy: 0.7531
precision: 0.7543
recall: 0.7506
f1: 0.7507

--- subgroup metrics ---
gender by race:
  0: 0.960
  1: 0.964
  2: 0.926
  3: 0.957
  4: 0.979
  5: 0.964
  6: 0.955
gender by age:
  0: 0.848
  1: 0.892
  2: 0.922
  3: 0.973
  4: 0.982
  5: 0.976
  6: 0.970
  7: 0.969
  8: 0.949
age by gender:
  0: 0.629
  1: 0.622
age by race:
  0: 0.652
  1: 0.610
  2: 0.615
  3: 0.608
  4: 0.652
  5: 0.616
  6: 0.642
race by gender:
  0: 0.754
  1: 0.752
race by age:
  0: 0.737
  1: 0.740
  2: 0.790
  3: 0.757
  4: 0.746
  5: 0.7